# Task 02 — Single-model baselines

## 🌌 Macrocosm photo-z — outlier study (tasks 2026-06-16)

**✅ GIVEN (do NOT re-derive this):** we already ran a **400k 5-fold cross-validation** with three
models (HGB, RF, MLP) and collected the galaxies that *all three* predict catastrophically out-of-fold
(|Δz/(1+z)| > 0.05). There are **6,974** such **hard** galaxies. Their objids are in
**`../hard_objids.csv`**. Treat this hard set as a **fixed input** — your job is to characterize / act
on it, not to re-find it from 400k.

**Catalog:** `gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet` (600k rows, 55 columns).
The SDSS **`-9999` sentinel** means "not measured" — always clean it to NaN first.

**Metric:** `σ_MAD = 1.4826 · median(|Δz − median(Δz)|)`, with `Δz = (z_pred − z_true)/(1+z)`;
an **outlier** is `|Δz| > 0.05`. σ_MAD is the headline metric, report **outlier rate** alongside it.

---

Establish the tabular baseline. Train four regressors and rank them by σ_MAD and outlier rate. These
numbers are the reference every later task compares against.

In [1]:
# === shared setup: load catalog, clean -9999, build the 16 features ===
import pandas as pd, numpy as np

CATALOG = "/Users/appalonovaa/code/Macrocosm/catalog_v1.parquet"
# Colab: from google.colab import auth; auth.authenticate_user()
# or download once: !gcloud storage cp $CATALOG . , then CATALOG = "catalog_v1.parquet"

FEATS = ["dered_u","dered_g","dered_r","dered_i","dered_z","g-r","u-g","r-i","i-z",
         "log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r",
         "fracDeV_r","conc_r"]

def load_features(path=CATALOG, n=None, seed=0):
    """Load catalog, clean the -9999 sentinel, build colors / log-sizes / conc.
    Returns (D, cat): D = features+redshift+objid (optionally subsampled), cat = full cleaned catalog."""
    cat = pd.read_parquet(path)
    num = cat.select_dtypes("number").columns
    cat[num] = cat[num].mask(cat[num] <= -100)                       # clean SDSS -9999
    for a, b in [("u","g"),("g","r"),("r","i"),("i","z")]:
        cat[f"{a}-{b}"] = (cat[f"dered_{a}"] - cat[f"dered_{b}"]).clip(-1, 4)
    for s in ["expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r"]:
        cat["log_"+s] = np.log1p(cat[s].clip(lower=0))
    cat["conc_r"] = cat["petroR90_r"] / cat["petroR50_r"].replace(0, np.nan)
    D = cat[FEATS + ["redshift","objid"]].replace([np.inf,-np.inf], np.nan).dropna()
    if n:
        D = D.sample(n, random_state=seed).reset_index(drop=True)
    return D, cat

def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    dz = (y_pred - y_true) / (1 + y_true)
    return {"MAE": round(float(np.mean(np.abs(y_pred-y_true))), 5),
            "sigma_MAD": round(float(smad(dz)), 5),
            "outlier_rate": round(float(np.mean(np.abs(dz) > 0.05)), 5)}

# the GIVEN hard set (6,974 objids from the 400k 5-fold CV)
HARD = set(pd.read_csv("../hard_objids.csv")["objid"])
print(f"hard set: {len(HARD)} galaxies")

hard set: 6974 galaxies


👇 Use these exact models / configs so everyone's numbers line up:

In [2]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def models():
    return {
        "LR":  make_pipeline(StandardScaler(), LinearRegression()),
        "RF":  RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=0),
        "HGB": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, early_stopping=True, random_state=0),
        "MLP": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(128,64,32), alpha=1e-4,
                   batch_size=1024, learning_rate_init=1e-3, early_stopping=True, n_iter_no_change=12,
                   max_iter=100, random_state=0)),
    }

❓ **Question (3-fold CV)** ❓

👇 Load 100k with `load_features(n=100000, seed=1)`. Run `KFold(3, shuffle=True, random_state=42)`. For each model collect out-of-fold predictions and compute `metrics(...)`. Print a table.

In [ ]:
from sklearn.base import clone

D, cat = load_features(n=100000, seed=1)
X, y = D[FEATS], D["redshift"]

kf = KFold(3, shuffle=True, random_state=42)

results = {}

for model_name, model in models().items():
    oof_predictions = np.full(len(y), np.nan)

    for train_idx, val_idx in kf.split(X):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]

        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)

        oof_predictions[val_idx] = fitted_model.predict(X_val)

    results[model_name] = metrics(y, oof_predictions)

results_df = pd.DataFrame(results).T
print(results_df)

         MAE  sigma_MAD  outlier_rate
LR   0.02489    0.02394       0.07676
RF   0.01633    0.01432       0.03059
HGB  0.01640    0.01460       0.03023
MLP  0.01670    0.01487       0.03240


❓ **Question (ranking)** ❓

👇 Which model has the lowest σ_MAD? The lowest outlier rate?

In [11]:
best_sigma_model = results_df["sigma_MAD"].idxmin()
best_outlier_model = results_df["outlier_rate"].idxmin()

print("Lowest sigma_MAD:", best_sigma_model)
print("Lowest outlier rate:", best_outlier_model)

Lowest sigma_MAD: RF
Lowest outlier rate: HGB


## 📝 Write your report

In [12]:
# === write_report: run this after you've filled in your results, it generates report.md ===
def write_report(title, results: dict, conclusion: str, path="report.md"):
    lines = [f"# {title}", "", "_Auto-generated by task.ipynb — fill `results` and `conclusion` above._", "",
             "## Results", ""]
    for k, v in results.items():
        lines.append(f"- **{k}**: {v}")
    lines += ["", "## Conclusion", "", conclusion, ""]
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print("wrote", path)

In [14]:
write_report(
    "Task 02 — Baselines",
    {
        "sigma_MAD per model": results_df["sigma_MAD"].to_dict(),
        "outlier_rate per model": results_df["outlier_rate"].to_dict(),
        "best model": best_sigma_model
    },
    "RF has the lowest sigma_MAD at 0.01432, while HGB has the lowest outlier rate at 0.03023. "
    "RF is better than HGB by 0.00028 in sigma_MAD, so the difference is small. "
    "Overall, RF is the best model based on the main metric, but RF and HGB perform very similarly."
)

wrote report.md


In [ ]:
# === Commit & push your results (run last; make sure you are on branch 2026.6.16) ===
# First time: git pull origin 2026.6.16   to get the latest.
!git add task.ipynb report.md && git commit -m "02-baseline-models task" && git push origin 2026.6.16

## 🔭 Go further (optional)

Play around with the data and the results you now have. If you find anything new — an unexpected pattern, a useful feature, a failure mode we missed — write it up and post it to our **YouTrack Knowledge Base** so the whole team benefits.